# Linear regression — Boomerang vs Sticky Boomerang

**What to expect from a healthy run:**

- **Plain Boomerang:** posterior means should match OLS/MAP. Posterior stds should match the Laplace-approximation stds (small — the posterior is concentrated). All P(=0) ≈ 0.
- **Sticky Boomerang:** signal coefficients recovered with roughly the same uncertainty as the plain version. Null coefficients have P(=0) > 0.5 and much smaller stds.

**Warning sign:** if the plain Boomerang posterior stds are all ≈ `prior_std` (e.g. all ~0.9), the sampler isn't informed by the data — likely stuck near the reference measure.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

import os
os.chdir('../..')

from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.samplers.StickyAutomaticBoomerangSampler import StickyAutomaticBoomerangSampler
from sazz.models.glm import make_linear_regression, make_kappa_vector_glm
from sazz.models.sticky_smoke_test import make_sparse_regression
from sazz.utils.sampling import resample_pdmp_path, resample_pdmp_path_sticky

## 1. Data

Sparse linear model: 3 true signals in 10 features, observation noise `σ = 0.3`.

In [ ]:
rng = np.random.default_rng(0)
N, D = 200, 10

X = rng.normal(size=(N, D))
beta_true = np.zeros(D)
beta_true[[0, 3, 7]] = [1.5, -2.0, 0.8]
intercept_true = 0.5
y = X @ beta_true + intercept_true + 0.3 * rng.normal(size=N)

X = (X - X.mean(0)) / X.std(0)

X_t = torch.tensor(X, dtype=torch.float64)
y_t = torch.tensor(y, dtype=torch.float64)

# True coefficient vector: [intercept, beta_1, ..., beta_D]
true_coefs = np.concatenate([[intercept_true], beta_true])
is_signal = true_coefs != 0
print(f"N={N}, D={D}, signals={is_signal.sum()}/{len(true_coefs)}")

## 2. Reference: OLS / MAP and HMC

A fitted OLS is our point-estimate sanity check. Posterior means from both samplers should land near these values.

In [ ]:
# OLS with intercept
X_aug = np.column_stack([np.ones(N), X])
ols_coefs = np.linalg.solve(X_aug.T @ X_aug, X_aug.T @ y)
resid = y - X_aug @ ols_coefs
sigma2_hat = (resid ** 2).sum() / (N - X_aug.shape[1])
ols_cov = sigma2_hat * np.linalg.inv(X_aug.T @ X_aug)
ols_stds = np.sqrt(np.diag(ols_cov))

print(f"{'coef':<8} {'true':>8} {'OLS':>8} {'OLS std':>9}")
for i, (t, m, s) in enumerate(zip(true_coefs, ols_coefs, ols_stds)):
    label = 'intercept' if i == 0 else f'β_{i}'
    star = ' *' if t != 0 else ''
    print(f"{label:<8} {t:>8.3f} {m:>8.3f} {s:>9.3f}{star}")

##

In [ ]:
import pymc as pm

with pm.Model() as linreg_model:
    # Match the priors in make_linear_regression
    intercept = pm.Normal('intercept', mu=0.0, sigma=10.0)
    betas = pm.Normal('betas', mu=0.0, sigma=1.0, shape=D)
    mu = intercept + X @ betas
    pm.Normal('y', mu=mu, sigma=0.5, observed=y)

    nuts_trace = pm.sample(
        draws=2000, tune=1000, chains=2,
        target_accept=0.9, progressbar=True, random_seed=0,
    )

# Flatten to match the Boomerang parameterisation: [intercept, β_1, ..., β_D]
nuts_intercept = nuts_trace.posterior['intercept'].values.reshape(-1)      # [chains * draws]
nuts_betas     = nuts_trace.posterior['betas'].values.reshape(-1, D)       # [chains * draws, D]
samples_nuts   = np.column_stack([nuts_intercept, nuts_betas])

print(f"NUTS: {samples_nuts.shape[0]} samples × {samples_nuts.shape[1]} coefficients")

## 3. Run samplers

In [ ]:
target = make_linear_regression(
    X_t, y_t,
    prior_std=1.0,
    intercept_prior_std=10.0,
    noise_std=0.5,
)

sampler = AutomaticBoomerangSampler(
    grad_target=target.grad_target, D=target.D, refresh_rate=1.0, thinning='pli',
)
sampler.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)

# Manual kappa: intercept never sticks, signals get small kappa (more likely zero)
# if they really are noise, nulls get small kappa too so shrinkage works
kappa = torch.full((target.D,), 1.0, dtype=torch.float64)
kappa[0] = 1e6  # intercept: never sticks

sampler_sticky = StickyAutomaticBoomerangSampler(
    grad_target=target.grad_target, D=target.D, refresh_rate=1.0,
    kappa=kappa, thinning='pli',
)
sampler_sticky.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)

In [ ]:
N_SKEL = 10_000
result = sampler.sample(N=N_SKEL, diagnostics=False)
result_sticky = sampler_sticky.sample(N=N_SKEL, diagnostics=False)

N_RESAMPLE = 50_000
BURNIN = 0.5

samples = resample_pdmp_path(
    result['positions'].cpu().numpy(),
    result['velocities'].cpu().numpy(),
    result['times'].cpu().numpy(),
    target.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE, burnin_frac=BURNIN,
)
samples_sticky = resample_pdmp_path_sticky(
    result_sticky['positions'].cpu().numpy(),
    result_sticky['velocities'].cpu().numpy(),
    result_sticky['times'].cpu().numpy(),
    target.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE, burnin_frac=BURNIN,
)
print(f"Resampled: {samples.shape[0]} samples × {samples.shape[1]} coefficients")

## 4. Comparison table

Each row is one coefficient. Compare the posterior means to `true` and `OLS`. For the plain Boomerang, the std column should match `OLS std`. For Sticky, `P(=0)` should be high for null coefs (true = 0.000) and ≈ 0 for signals.

In [ ]:
means = samples.mean(0); stds = samples.std(0)
means_s = samples_sticky.mean(0); stds_s = samples_sticky.std(0)
means_n = samples_nuts.mean(0); stds_n = samples_nuts.std(0)
p_zero = (np.abs(samples_sticky) < 1e-8).mean(0)

header = (f"{'coef':<10} {'true':>7} {'OLS':>7}"
          f"  |  {'NUTS μ':>7} {'NUTS σ':>7}"
          f"  |  {'Boom μ':>7} {'Boom σ':>7}"
          f"  |  {'Stky μ':>7} {'Stky σ':>7} {'P(=0)':>6}")
print(header)
print('-' * len(header))
for i in range(len(true_coefs)):
    label = 'intercept' if i == 0 else f'β_{i}'
    star = ' *' if is_signal[i] else ''
    print(f"{label:<10} {true_coefs[i]:>7.3f} {ols_coefs[i]:>7.3f}"
          f"  |  {means_n[i]:>7.3f} {stds_n[i]:>7.3f}"
          f"  |  {means[i]:>7.3f} {stds[i]:>7.3f}"
          f"  |  {means_s[i]:>7.3f} {stds_s[i]:>7.3f} {p_zero[i]:>6.2f}{star}")

print(f"\nRMSE vs NUTS mean:")
print(f"  Boomerang: {np.sqrt(((means - means_n)**2).mean()):.4f}"
      f"   Sticky: {np.sqrt(((means_s - means_n)**2).mean()):.4f}")
print(f"Std ratio (sampler/NUTS, averaged):")
print(f"  Boomerang: {(stds / stds_n).mean():.2f}  (want ≈ 1.0)"
      f"   Sticky (signals): {(stds_s[is_signal] / stds_n[is_signal]).mean():.2f}")

## 5. Calibration plot

Posterior mean ± 2 std for each coefficient. Black crosses are the true values. Signals are highlighted. You want the true value inside the interval; for Sticky you also want the intervals for null coefficients to visibly collapse toward 0.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
idx = np.arange(len(true_coefs))
offset = 0.22

ax.errorbar(idx - offset, means_n, yerr=2 * stds_n, fmt='D', color='C2',
            label='NUTS (mean ± 2σ)', capsize=3, markersize=5)
ax.errorbar(idx, means, yerr=2 * stds, fmt='o', color='C0',
            label='Boomerang (mean ± 2σ)', capsize=3, markersize=5)
ax.errorbar(idx + offset, means_s, yerr=2 * stds_s, fmt='s', color='C1',
            label='Sticky (mean ± 2σ)', capsize=3, markersize=5)
ax.scatter(idx, true_coefs, marker='x', color='k', s=80, linewidths=2,
           label='true', zorder=5)

for i in np.where(is_signal)[0]:
    ax.axvspan(i - 0.45, i + 0.45, color='gold', alpha=0.15)

ax.axhline(0, color='grey', lw=0.5)
ax.set_xticks(idx)
ax.set_xticklabels(['int.'] + [f'β_{i}' for i in range(1, len(true_coefs))])
ax.set_ylabel('coefficient value')
ax.set_title('Posterior intervals — NUTS (green) is the gold-standard reference')
ax.legend(loc='best', frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

## 6. Trace plot for one signal

Quick mixing check for the strongest signal coefficient. The chain should look like stationary noise around the OLS value, not drifting.

In [ ]:
# Pick the strongest true signal
j = np.argmax(np.abs(true_coefs))

fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, samp, name, color in [
    (axes[0], samples_nuts, 'NUTS', 'C2'),
    (axes[1], samples, 'Boomerang', 'C0'),
    (axes[2], samples_sticky, 'Sticky', 'C1'),
]:
    ax.plot(samp[:, j], lw=0.4, color=color)
    ax.axhline(true_coefs[j], color='k', lw=1, ls='--',
               label=f'true = {true_coefs[j]:.2f}')
    ax.set_title(f'{name}: β_{j} trace')
    ax.set_xlabel('sample index')
    ax.legend(loc='best', fontsize=8, frameon=False)
axes[0].set_ylabel(f'β_{j}')
plt.tight_layout()
plt.show()

In [ ]:
# Pick the strongest true signal
j = np.argmin(np.abs(true_coefs))

fig, axes = plt.subplots(1, 3, figsize=(13, 3), sharey=True)
for ax, samp, name, color in [
    (axes[0], samples_nuts, 'NUTS', 'C2'),
    (axes[1], samples, 'Boomerang', 'C0'),
    (axes[2], samples_sticky, 'Sticky', 'C1'),
]:
    ax.plot(samp[:, j], lw=0.4, color=color)
    ax.axhline(true_coefs[j], color='k', lw=1, ls='--',
               label=f'true = {true_coefs[j]:.2f}')
    ax.set_title(f'{name}: β_{j} trace')
    ax.set_xlabel('sample index')
    ax.legend(loc='best', fontsize=8, frameon=False)
axes[0].set_ylabel(f'β_{j}')
plt.tight_layout()
plt.show()

## CORRELATED MODEL

In [ ]:
rng = np.random.default_rng(0)
N, D = 200, 10

# AR(1)-correlated features: Cov[i,j] = rho^|i-j|
rho = 0.8
idx = np.arange(D)
cov = rho ** np.abs(idx[:, None] - idx[None, :])
X = rng.multivariate_normal(np.zeros(D), cov, size=N)

beta_true = np.zeros(D)
beta_true[[0, 3, 7]] = [1.5, -2.0, 0.8]
intercept_true = 0.5
y = X @ beta_true + intercept_true + 0.3 * rng.normal(size=N)

X = (X - X.mean(0)) / X.std(0)

X_t = torch.tensor(X, dtype=torch.float64)
y_t = torch.tensor(y, dtype=torch.float64)

true_coefs = np.concatenate([[intercept_true], beta_true])
is_signal = true_coefs != 0

# Quick diagnostic: look at the empirical correlation
print(f"N={N}, D={D}, signals={is_signal.sum()}/{len(true_coefs)}")
print(f"Target AR(1) correlation: rho={rho}")
print(f"Empirical off-diagonal correlations (max abs): "
      f"{np.abs(np.corrcoef(X.T) - np.eye(D)).max():.3f}")

In [ ]:
# OLS with intercept
X_aug = np.column_stack([np.ones(N), X])
ols_coefs = np.linalg.solve(X_aug.T @ X_aug, X_aug.T @ y)
resid = y - X_aug @ ols_coefs
sigma2_hat = (resid ** 2).sum() / (N - X_aug.shape[1])
ols_cov = sigma2_hat * np.linalg.inv(X_aug.T @ X_aug)
ols_stds = np.sqrt(np.diag(ols_cov))

print(f"{'coef':<8} {'true':>8} {'OLS':>8} {'OLS std':>9}")
for i, (t, m, s) in enumerate(zip(true_coefs, ols_coefs, ols_stds)):
    label = 'intercept' if i == 0 else f'β_{i}'
    star = ' *' if t != 0 else ''
    print(f"{label:<8} {t:>8.3f} {m:>8.3f} {s:>9.3f}{star}")

In [ ]:
import pymc as pm

with pm.Model() as linreg_model:
    # Match the priors in make_linear_regression
    intercept = pm.Normal('intercept', mu=0.0, sigma=10.0)
    betas = pm.Normal('betas', mu=0.0, sigma=1.0, shape=D)
    mu = intercept + X @ betas
    pm.Normal('y', mu=mu, sigma=0.5, observed=y)

    nuts_trace = pm.sample(
        draws=2000, tune=1000, chains=2,
        target_accept=0.9, progressbar=True, random_seed=0,
    )

# Flatten to match the Boomerang parameterisation: [intercept, β_1, ..., β_D]
nuts_intercept = nuts_trace.posterior['intercept'].values.reshape(-1)      # [chains * draws]
nuts_betas     = nuts_trace.posterior['betas'].values.reshape(-1, D)       # [chains * draws, D]
samples_nuts   = np.column_stack([nuts_intercept, nuts_betas])

print(f"NUTS: {samples_nuts.shape[0]} samples × {samples_nuts.shape[1]} coefficients")

In [ ]:
target = make_linear_regression(
    X_t, y_t,
    prior_std=1.0,
    intercept_prior_std=10.0,
    noise_std=0.5,
    diagonal_only=True
)

sampler = AutomaticBoomerangSampler(
    grad_target=target.grad_target, D=target.D, refresh_rate=1.0, thinning='pli',
)
sampler.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)

# Manual kappa: intercept never sticks, signals get small kappa (more likely zero)
# if they really are noise, nulls get small kappa too so shrinkage works
kappa = torch.full((target.D,), 1.0, dtype=torch.float64)
kappa[0] = 1e6  # intercept: never sticks

sampler_sticky = StickyAutomaticBoomerangSampler(
    grad_target=target.grad_target, D=target.D, refresh_rate=1.0,
    kappa=kappa, thinning='pli',
)
sampler_sticky.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)

In [ ]:
N_SKEL = 10_000
result = sampler.sample(N=N_SKEL, diagnostics=False)
result_sticky = sampler_sticky.sample(N=N_SKEL, diagnostics=False)

N_RESAMPLE = 50_000
BURNIN = 0.1

samples = resample_pdmp_path(
    result['positions'].cpu().numpy(),
    result['velocities'].cpu().numpy(),
    result['times'].cpu().numpy(),
    target.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE, burnin_frac=BURNIN,
)
samples_sticky = resample_pdmp_path_sticky(
    result_sticky['positions'].cpu().numpy(),
    result_sticky['velocities'].cpu().numpy(),
    result_sticky['times'].cpu().numpy(),
    target.x_ref.cpu().numpy(),
    N_resample=N_RESAMPLE, burnin_frac=BURNIN,
)
print(f"Resampled: {samples.shape[0]} samples × {samples.shape[1]} coefficients")

In [ ]:
means = samples.mean(0); stds = samples.std(0)
means_s = samples_sticky.mean(0); stds_s = samples_sticky.std(0)
means_n = samples_nuts.mean(0); stds_n = samples_nuts.std(0)
p_zero = (np.abs(samples_sticky) < 1e-8).mean(0)

header = (f"{'coef':<10} {'true':>7} {'OLS':>7}"
          f"  |  {'NUTS μ':>7} {'NUTS σ':>7}"
          f"  |  {'Boom μ':>7} {'Boom σ':>7}"
          f"  |  {'Stky μ':>7} {'Stky σ':>7} {'P(=0)':>6}")
print(header)
print('-' * len(header))
for i in range(len(true_coefs)):
    label = 'intercept' if i == 0 else f'β_{i}'
    star = ' *' if is_signal[i] else ''
    print(f"{label:<10} {true_coefs[i]:>7.3f} {ols_coefs[i]:>7.3f}"
          f"  |  {means_n[i]:>7.3f} {stds_n[i]:>7.3f}"
          f"  |  {means[i]:>7.3f} {stds[i]:>7.3f}"
          f"  |  {means_s[i]:>7.3f} {stds_s[i]:>7.3f} {p_zero[i]:>6.2f}{star}")

print(f"\nRMSE vs NUTS mean:")
print(f"  Boomerang: {np.sqrt(((means - means_n)**2).mean()):.4f}"
      f"   Sticky: {np.sqrt(((means_s - means_n)**2).mean()):.4f}")
print(f"Std ratio (sampler/NUTS, averaged):")
print(f"  Boomerang: {(stds / stds_n).mean():.2f}  (want ≈ 1.0)"
      f"   Sticky (signals): {(stds_s[is_signal] / stds_n[is_signal]).mean():.2f}")